# 04 — Avaliação Comparativa (CNN-A vs. CNN-B)

Compara os dois modelos com métricas quantitativas, matrizes de confusão, sobreposição de curvas e análise qualitativa de erros. Requer que os notebooks 02 e 03 já tenham sido executados (pesos e históricos salvos).

In [ ]:
# >>> NO GOOGLE COLAB: rode ESTA célula primeiro (em ambiente local, pule) <<<
# Clona o repositório. NÃO instalamos requirements.txt no Colab: ele já traz
# TensorFlow, Keras, numpy, pandas, scikit-learn, matplotlib, seaborn e
# tensorflow-datasets em versões compatíveis entre si. Reinstalar versões
# fixas rebaixa o ml_dtypes/numpy e quebra o JAX do Colab.
# !git clone https://github.com/SEU_USUARIO/gs-cnn-eurosat.git
# %cd gs-cnn-eurosat

In [ ]:
import sys, os
# Resolve a raiz do projeto de forma robusta (local OU Colab, com ou sem
# restart de runtime). Procura uma pasta que contenha 'src/'.
candidatos = [os.getcwd(), os.path.dirname(os.getcwd()), '/content/gs-cnn-eurosat']
ROOT = next((p for p in candidatos if os.path.isdir(os.path.join(p, 'src'))), os.getcwd())
os.chdir(ROOT)  # paths relativos (models/, reports/) consistentes
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print('Project root:', ROOT)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model
from src.data_loader import load_raw_data, make_splits, make_tf_datasets, CLASS_NAMES
from src.train import load_history
from src.evaluate import (get_predictions, text_report, report_dataframe,
                          plot_confusion_matrix, compare_training_curves,
                          plot_misclassified)

## Dados de teste (mesma divisão — seed=42)

In [ ]:
images, labels = load_raw_data()
splits = make_splits(images, labels, seed=42)
_, _, test_ds = make_tf_datasets(splits, batch_size=64)
x_test, y_test = splits['test']

## Carregar modelos e históricos

In [ ]:
model_a = load_model('models/cnn_a_best.keras')
model_b = load_model('models/cnn_b_best.keras')
hist_a = load_history('reports/cnn_a_history.json')
hist_b = load_history('reports/cnn_b_history.json')

## Predições no conjunto de teste

In [ ]:
yt_a, yp_a, pr_a = get_predictions(model_a, test_ds)
yt_b, yp_b, pr_b = get_predictions(model_b, test_ds)

## Relatório de classificação (precision / recall / f1 por classe)

In [ ]:
print('=== CNN-A (Baseline) ===  acurácia:', round(accuracy_score(yt_a, yp_a), 4))
print(text_report(yt_a, yp_a, CLASS_NAMES))
print('=== CNN-B (Refinada) ===  acurácia:', round(accuracy_score(yt_b, yp_b), 4))
print(text_report(yt_b, yp_b, CLASS_NAMES))

## Matrizes de confusão (normalizadas)

In [ ]:
plot_confusion_matrix(yt_a, yp_a, CLASS_NAMES, title='CNN-A',
                      save_path='reports/figures/cm_cnn_a.png')
plot_confusion_matrix(yt_b, yp_b, CLASS_NAMES, title='CNN-B',
                      save_path='reports/figures/cm_cnn_b.png')

## Curvas de validação sobrepostas

In [ ]:
compare_training_curves(hist_a, hist_b,
                        save_path='reports/figures/comparacao_curvas.png')

## Análise qualitativa — erros da CNN-B
Imagens do conjunto de teste classificadas incorretamente, com classe real, classe predita e confiança.

In [ ]:
plot_misclassified(x_test, yt_b, yp_b, pr_b, CLASS_NAMES, n=15,
                   save_path='reports/figures/erros_cnn_b.png')

## Tabela comparativa final

In [ ]:
tabela = pd.DataFrame({
    'Modelo': ['CNN-A (Baseline)', 'CNN-B (Refinada)'],
    'Parâmetros': [model_a.count_params(), model_b.count_params()],
    'Acc. validação (melhor)': [round(max(hist_a['val_accuracy']), 4),
                                 round(max(hist_b['val_accuracy']), 4)],
    'Acc. teste': [round(accuracy_score(yt_a, yp_a), 4),
                   round(accuracy_score(yt_b, yp_b), 4)],
})
tabela

## Análise técnica

*Preencha com base nos resultados reais acima. Estrutura sugerida:*

- **Qual modelo venceu e por quê:** a CNN-B deve superar a CNN-A em acurácia de teste e, principalmente, no gap treino–validação (generalização).
- **Efeito de cada técnica:** relacione a melhora à augmentation (robustez a variações), BatchNorm (treino estável), Dropout (regularização) e GAP (menos parâmetros, menos overfitting).
- **Confusões mais comuns (matriz):** classes de vegetação tendem a se confundir entre si — *AnnualCrop ↔ PermanentCrop ↔ HerbaceousVegetation ↔ Pasture* — por similaridade de textura/cor. Comente o que a matriz real mostrou.
- **Padrão dos erros:** descreva o que se observa nas imagens mal classificadas (ex.: confusões plausíveis vs. erros grosseiros).
- **Eficiência:** destaque que a CNN-B atinge desempenho superior com **menos** parâmetros que o baseline.